# 01 · Análisis exploratorio (EDA) — C-MAPSS
**TT 2027-A190 · CRISP-DM fases 2 (comprensión de los datos) y 3 (preparación)**

Objetivos: 

1. Caracterizar los subconjuntos FD001–FD004.
2. Identificar sensores constantes o irrelevantes.
3. Justificar la normalización por régimen de operación.
4. Visualizar la degradación y la etiqueta de RUL lineal por tramos.
5. Verificar la forma de las ventanas para la LSTM.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # raíz del repositorio

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import (DATA_RAW, N_REGIMES, OP_COLS, SENSOR_COLS, SENSOR_DESCRIPTIONS,
                        LITERATURE_SENSORS_FD001)
from src.ingestion import load_subset
from src.preprocessing import Preprocessor, add_rul, build_training_windows

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
SUBSET = "FD001"
data = load_subset(SUBSET, DATA_RAW)
print(f"{SUBSET}: {data.train['unit'].nunique()} motores de entrenamiento, "
      f"{data.test['unit'].nunique()} de prueba")

## 1. Panorama del subconjunto

In [ ]:
life = data.train.groupby("unit")["cycle"].max()
test_len = data.test.groupby("unit")["cycle"].max()
summary = pd.DataFrame({
    "vida (train)": life.describe(),
    "ciclos observados (test)": test_len.describe(),
}).round(1)
summary

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].hist(life, bins=20, color="tab:blue", edgecolor="white")
ax[0].set(title="Vida útil por motor (entrenamiento)", xlabel="Ciclos hasta la falla", ylabel="Motores")
ax[1].hist(test_len, bins=20, color="tab:orange", edgecolor="white")
ax[1].set(title="Longitud de trayectorias truncadas (prueba)", xlabel="Ciclos observados")
plt.tight_layout(); plt.show()

## 2. Regímenes de operación
FD001/FD003 operan en una sola condición; FD002/FD004 en seis (Saxena et al., 2008). Los ajustes operacionales (`op1`, `op2`, `op3`) forman grupos discretos, y los sensores cambian de escala según el régimen: por eso se normaliza **por régimen** y no con estadísticas globales.

In [ ]:
multi = "FD002" if (DATA_RAW / "train_FD002.txt").exists() else SUBSET
d2 = load_subset(multi, DATA_RAW)
pre_m = Preprocessor(n_regimes=N_REGIMES[multi]).fit(d2.train)
reg = pre_m.assign_regimes(d2.train)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sc = ax[0].scatter(d2.train["op1"], d2.train["op2"], c=reg, s=3, cmap="tab10")
ax[0].set(title=f"{multi}: ajustes operacionales coloreados por régimen", xlabel="op1", ylabel="op2")
for r in range(N_REGIMES[multi]):
    m = reg == r
    ax[1].hist(d2.train.loc[m, "s3"], bins=40, alpha=0.7, label=f"régimen {r}")
ax[1].set(title=f"{multi}: sensor s3 (T30) separado por régimen", xlabel="s3")
ax[1].legend(fontsize=7); plt.tight_layout(); plt.show()

## 3. Sensores constantes o irrelevantes
Un sensor se descarta si su desviación estándar **dentro de cada régimen** es menor que `std_tol` en todos los regímenes. Se contrasta con el conjunto de 14 sensores que suele conservarse en FD001 en la literatura.

In [ ]:
pre = Preprocessor(n_regimes=N_REGIMES[SUBSET]).fit(data.train)
std_table = data.train[SENSOR_COLS].std(ddof=0).rename("std global").to_frame()
std_table["descripción"] = std_table.index.map(SENSOR_DESCRIPTIONS)
std_table["conservado"] = std_table.index.isin(pre.feature_cols_)
print("Descartados:", pre.dropped_cols_)
print("Conservados:", pre.feature_cols_)
if SUBSET == "FD001":
    print("Coinciden con la lista de la literatura:", set(pre.feature_cols_) == set(LITERATURE_SENSORS_FD001))
    print("Diferencias:", set(pre.feature_cols_) ^ set(LITERATURE_SENSORS_FD001))
std_table.round(4)

## 4. Degradación en el tiempo (valores normalizados)

In [ ]:
train = add_rul(data.train, cap=125)
norm = pre.transform(train)
units = sorted(norm["unit"].unique())[:5]
show = pre.feature_cols_[:6]
fig, axes = plt.subplots(len(show), 1, figsize=(10, 2.0 * len(show)), sharex=True)
for ax, s in zip(axes, show):
    for u in units:
        g = norm[norm["unit"] == u]
        ax.plot(g["rul"].max() - g["rul"], g[s], lw=1, label=f"motor {u}")
    ax.set_ylabel(s)
axes[0].legend(ncol=5, fontsize=7); axes[-1].set_xlabel("Ciclo de operación")
plt.tight_layout(); plt.show()

In [ ]:
corr = norm[pre.feature_cols_ + ["rul"]].corr(method="spearman")["rul"].drop("rul").sort_values()
ax = corr.plot.barh(figsize=(7, 5), color="tab:green")
ax.set(title=f"{SUBSET}: correlación de Spearman sensor–RUL", xlabel="ρ"); plt.tight_layout(); plt.show()

## 5. Etiqueta de RUL lineal por tramos
Se asume que, mientras el motor está sano, la degradación es despreciable; la RUL se limita a un tope (`rul_cap`). El valor óptimo debe seleccionarse con validación (ver notebook 02) y justificarse con la literatura.

In [ ]:
u = units[0]
g = train[train["unit"] == u]
plt.figure(figsize=(8, 3.6))
plt.plot(g["cycle"], g["rul"], label="RUL lineal")
plt.plot(g["cycle"], g["rul_target"], label="RUL por tramos (tope = 125)")
plt.xlabel("Ciclo"); plt.ylabel("RUL"); plt.legend(); plt.title(f"Motor {u}"); plt.show()

## 6. Ventanas temporales para la LSTM

In [ ]:
WINDOW = 30
X, y, un = build_training_windows(norm, pre.feature_cols_, WINDOW)
print(f"X: {X.shape}  (muestras, ciclos por ventana, sensores)")
print(f"y: {y.shape}  | rango de etiqueta: [{y.min():.0f}, {y.max():.0f}]")
print("Ventanas por motor (primeros 5):")
pd.Series(un).value_counts().sort_index().head()

## Conclusiones del EDA (completar con los resultados reales)
1. Sensores descartados y su coincidencia con la literatura.
2. Justificación de la normalización por régimen (FD002/FD004).
3. Rango de longitudes de trayectoria y su efecto en el tamaño de ventana (motores de prueba muy cortos requieren relleno).
4. Candidatos a `rul_cap` y `window` para la búsqueda en cuadrícula.

**Referencias**: Saxena, A., Goebel, K., Simon, D. y Eklund, N. (2008). Damage propagation modeling for aircraft engine run-to-failure simulation. *2008 International Conference on Prognostics and Health Management*. https://doi.org/10.1109/PHM.2008.4711414